# **walter**

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
from IPython.display import display


project_root = Path().resolve().parent
sys.path.append(str(project_root))

## **nomenclature and terminologies**

The dataset $\mathcal{D}_{\text{raw}}$ (represented as `D_raw` in the source code) refers to the raw Philippine human-drug registry, which is freely available as a `.csv` file at [https://verification.fda.gov.ph/drug_productslist.php](https://verification.fda.gov.ph/drug_productslist.php).

The intermediate dataset, $\mathcal{D}_{\text{clean}}$ (`D_clean`), is the result of passing $\mathcal{D}_{\text{raw}}$ through the preprocessing pipeline. 

The final dataset, $\mathcal{D}_{\text{train}}$ (`D_train`), is used to train a weighted sum of similarity measures via a genetic algorithm. It consists of ordered pairs of drugs formed from the cleaned registry, such that every pair $(x, y) \in \mathcal{D}_{\text{clean}} \times \mathcal{D}_{\text{clean}}$. This training dataset is partitioned into two disjoint subsets:

* **$P \subset \mathcal{D}_{\text{train}}$** (`P`) is the set of known positives, consisting of ordered drug pairs that are manually verified as LASA.
* **$U \subset \mathcal{D}_{\text{train}}$** (`U`) is the unlabeled noise set, consisting of randomly paired drugs from $\mathcal{D}_{\text{clean}}$. $U$ acts as the noise class (its true labels are unknown), so it may contain undetected LASA pairs.

Furthermore, $|U| \gg |P|$, $P \cap U = \emptyset$, and $P \cup U = \mathcal{D}_{\text{train}}$.

## **preprocessing**

The first step is to preprocess (load, validate, clean) the FDA human drug registry dataset to construct our $\mathcal{D}_\text{clean}$ dataset.

Ensure that the FDA human drug registry dataset exists anywhere starting from the root folder and has the same filename defined by `PRIMARY_FNAME`.

The code for this section is located at [`/src/preprocessing.py`](/src/preprocessing.py).

In [2]:
import src.preprocessing as pre

The function `master_maker` is the coordinator function that performs data loading, validation, cleaning, and reporting. 

In [3]:
D_clean = pre.master_maker(sort=True, save=True)

<walter> Validation Warning: Found 2986 missing (NaN) values.
<walter> Validation Warning: Found 9772 exact duplicate rows.
<walter> Validation Warning: Dataset has entries that contain non-ASCII characters. In total, there are 100 entries.
<walter> Cleaning Report: 
  - Total rows dropped during cleaning: 9791
  - Non-ASCII entries sanitized/removed: 100 (Out of 100 original dirty entries).
  - Samples of modified text (repr format):
    Original: 'Quelicin®'
    Cleaned:  'Quelicin'
    ---
    Original: 'Betaloc®'
    Cleaned:  'Betaloc'
    ---
    Original: 'Budecort®'
    Cleaned:  'Budecort'
    ---
    Original: 'Tenormin®'
    Cleaned:  'Tenormin'
    ---
    Original: 'Zestril®'
    Cleaned:  'Zestril'
    ---


Let's look at the info of the dataset.

In [4]:
D_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 22838 entries, 0 to 22837
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Brand Name  22838 non-null  str  
dtypes: str(1)
memory usage: 178.6 KB


Then, the head of the dataset.

In [5]:
D_clean.head()

,Brand Name
0,0.9% NaCl-Sapher
1,0.9% Sodchlorsaph
2,1 Ceeplus
3,1000Vc
4,2-Gen


Finally, let's look at a slice of 10 entries in the dataset by using the `get_rand_entries()` function.

In [ ]:
display(pre.get_rand_entries(df=D_clean, count=10))

,Brand Name
10261,Icard IV
10262,ICARE
10263,Icelax
10264,Icon
10265,Iconagel
10266,Icox
10267,Icoxib
10268,Icumax
10269,Icunes
10270,Idacio


,Brand Name
0,0.9% NaCl-Sapher
1,0.9% Sodchlorsaph
2,1 Ceeplus
3,1000Vc
4,2-Gen
...,...
22833,Zytrex
22834,Zyvax
22835,Zyvosal
22836,Zyvox


## **true LASA pairs**

Now that we have the $\mathcal{D}_\text{clean}$ we can now proceed with constructing $\mathcal{D}_\text{train}$. We will prioritize constructing the subset of true LASA pairs, or the set $P$.

The code for this section is located at [`/src/proposer/`](/src/proposer/).

In [7]:
from src.proposer import propose, Model
from src.preprocessing import table_to_string

### **prompt testing ground using Llama 3.3**

In [ ]:
model = Model.HF

iters = 1
P_hf = []

for _ in range(iters):
    sample_df: pd.DataFrame = D_clean.sample(n=1, random_state=None)
    remaining_df: pd.DataFrame = D_clean.drop(sample_df.index)

    sample_str: str = table_to_string(sample_df)
    remaining_str: str = table_to_string(remaining_df)
    
    proposals = propose(sample_str, remaining_str, model)

    P_hf.append((sample_str, proposals))

print(P_hf)


<walter> Proposing LASA for: Rylin.


/home/zrgnt/walter/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


ValueError: Could not load model deepseek-ai/DeepSeek-R1-Distill-Llama-8B with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>, <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>). See the original errors:

while loading with AutoModelForCausalLM, an error is thrown:
Traceback (most recent call last):
  File "/home/zrgnt/walter/.venv/lib/python3.11/site-packages/transformers/pipelines/base.py", line 232, in load_model
    model = model_class.from_pretrained(model, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zrgnt/walter/.venv/lib/python3.11/site-packages/transformers/models/auto/auto_factory.py", line 394, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zrgnt/walter/.venv/lib/python3.11/site-packages/transformers/modeling_utils.py", line 4089, in from_pretrained
    device_map = check_and_set_device_map(device_map)  

## **noise pairs**

Now that we have $P$, we can now complete constructing $\mathcal{D}_\text{train}$ by constructing the set $U$ or the unlabeled noise set.

The code for this section is located at [`/src/noise.py`](/src/noise.py)

In [ ]:
import src.noise as noise

In [ ]:
from src.utils import finder

sample_file: Path = finder(fname="sample_true_lasa.csv")
sample_P: pd.DataFrame = pd.read_csv(filepath_or_buffer=sample_file)

lasa_set = noise.get_lasa_set(true_df=sample_P)

assert len(lasa_set) == 20

sample_U = noise.make_noise(fda_df=D_clean, true_df=sample_P, n=3)

sample_U

,Drug Name 1,Drug Name 2
0,Vendicom,Tgxime
1,Vendicom,Verlene-20
2,Tgxime,Vendicom
3,Tgxime,Verlene-20
4,Verlene-20,Vendicom
5,Verlene-20,Tgxime


## **assembling the training dataset**

Now that both subsets are complete. Assembling $\mathcal{D}_\text{train}$ is simply a concatenation of $P$ and $U$. 

In [ ]:
...

Ellipsis